# 04 — Specific risk Δ(t)

> **Universe build** (`ryan/universe`). Same model code as `ryan/sample`; the
> universe, tables and history come from `config.py` — profile set by the
> `BARRA_UNIVERSE` environment variable (default `r1000`). See the README.

**Step 6 of the model.** One specific variance per (asset, forecast date), the
diagonal of

$$\Sigma(t) = X(t)\,F(t)\,X(t)^\top + \Delta(t)$$

Same convention as 03: `Δ(t)` uses $u(i,s)$ for $s \le t$ and forecasts the
variance of $u(i,t+1)$. An asset gets a forecast at *t* if it has an exposure row
at *t*, meaning it could be held at the end of month *t*.

Reads `ryan_<profile>_specific_returns`, `ryan_<profile>_exposures`, and the forecast
dates from `ryan_<profile>_factor_covariance`, so the two halves of Σ always exist
at the same dates. Writes `ryan_<profile>_specific_risk` keyed `(id, forecast_date)`.

| stage | what it does | why |
|---|---|---|
| `raw` | decay-weighted variance of the asset's own `u` over 60 months | the asset's own history |
| `shrunk` | Bayesian shrinkage of the **vol** toward its size decile's cap-weighted mean | the extreme estimates are extreme partly *because of* estimation error |
| `final` | × λ_S², the specific volatility regime adjustment | the whole cross-section's specific risk level moves |

Newey-West is off (`SPECIFIC_NW_LAGS = 0`). Stage 2 measures why.

In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())

# ryan/universe/config.py -- imported FIRST: it shadows the sample's config for
# every shared module and puts ryan/sample (where the model code lives) on the path.
import config
assert config.PROFILE != "sample", "wrong config on the path — run this notebook from ryan/universe"
print(config.describe())

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import db
import riskmath
import specific_risk

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

INK, ACCENT, MUTED, GREEN = "#2a78d6", "#e34948", "#8a8a83", "#1baf7a"
STAGE_COLORS = {"specific_var_raw": MUTED, "specific_var_shrunk": GREEN, "specific_var": INK}
STAGE_NAMES = {"specific_var_raw": "raw", "specific_var_shrunk": "shrunk", "specific_var": "final"}


def tidy(ax):
    ax.grid(alpha=.25, lw=.6)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


REBUILD_SPECIFIC = True

print(f"window {config.SPECIFIC_WINDOW}, half-life {config.SPECIFIC_HALFLIFE}, "
      f"NW lags {config.SPECIFIC_NW_LAGS}, min obs {config.SPECIFIC_MIN_OBS}")
print(f"shrinkage q {config.SHRINK_Q} over {config.SHRINK_BUCKETS} size buckets; "
      f"VRA half-life {config.SPECIFIC_VRA_HALFLIFE}")

In [ ]:
engine = db.get_engine()
db.check_connection(engine)

---
## Stage 1 — Load u(i,t), exposures, and F's forecast dates

In [ ]:
for t in (config.TBL_SPECIFIC_RETURNS, config.TBL_EXPOSURES, config.TBL_FACTOR_COV_DIAG):
    if not db.table_exists(engine, t):
        raise RuntimeError(f"{t} is missing — run 02 and 03 first.")

U = db.read_table(config.TBL_SPECIFIC_RETURNS, engine)
U["eom"] = pd.to_datetime(U["eom"])
EX = db.read_table(config.TBL_EXPOSURES, engine)
EX["eom"] = pd.to_datetime(EX["eom"])
dates = pd.DatetimeIndex(pd.to_datetime(
    db.read_table(config.TBL_FACTOR_COV_DIAG, engine)["forecast_date"]).sort_values())

u_wide = U.pivot(index="eom", columns="id", values="specific_return")
print(f"u(i,t): {len(U):,} rows, {U['id'].nunique()} securities, "
      f"{U['eom'].min().date()} → {U['eom'].max().date()}")
print(f"forecast dates from F: {len(dates)}, {dates[0].date()} → {dates[-1].date()}")

---
## Stage 2 — Does Newey-West help Δ? Measured, not assumed

Pooled over every (asset, month) forecast, $z = u(i,t+1) / \sigma_i(t)$. The
lag-1 autocorrelation of `u` is mildly negative (mean −0.047, 66% of names), so
NW *lowers* each variance estimate. The question is whether realized specific
risk is lower too.

Two statistics, because specific returns are fat-tailed. The **std** of z is
the bias statistic. The **robust** one is 1.4826 × median|z|, which is 1 for a
normal and tells you what the typical month looks like when a few +2000%
months are excluded.

In [ ]:
nxt = dict(zip(dates[:-1], dates[1:]))
u_long = u_wide.stack().rename("u_next").reset_index().rename(columns={"eom": "next"})


def realized(delta):
    d = delta.assign(next=delta["forecast_date"].map(nxt)).dropna(subset=["next"])
    return d.merge(u_long, on=["next", "id"], how="inner")


def score(m, col):
    z = m["u_next"] / np.sqrt(m[col])
    ok = np.isfinite(z)
    q = pd.qcut(m.loc[ok, col], 5, labels=False)
    by_q = z[ok].groupby(q).std()
    return {
        "bias (std z)": z[ok].std(),
        "robust (MAD)": 1.4826 * np.median(np.abs(z[ok])),
        "median B_t": z[ok].groupby(m.loc[ok, "forecast_date"]).std().median(),
        "lowest-vol quintile": by_q.iloc[0],
        "highest-vol quintile": by_q.iloc[-1],
        "quintile spread": by_q.iloc[0] - by_q.iloc[-1],
    }


rows = {}
for lags in (0, None):
    d = specific_risk.estimate(U, EX, dates, lags=lags, verbose=False)
    m = realized(d)
    label = "NW lags 0" if lags == 0 else f"NW lags {riskmath.newey_west_lags(config.SPECIFIC_WINDOW)} (rule of thumb)"
    for col in STAGE_NAMES:
        rows[(label, STAGE_NAMES[col])] = score(m, col)
display(pd.DataFrame(rows).T.round(3))
print(f"95% band for a bias statistic over {len(m):,} pooled forecasts is ±{riskmath.bias_ci(len(m))[1] - 1:.3f};")
print("that band is misleadingly tight (the forecasts are not independent across names), so read the")
print("comparisons across rows, not against the band.")

On the sample, NW added ~0.1 of understatement at the raw stage and VRA only
partly repaired it. That is why `SPECIFIC_NW_LAGS = 0`. The rows above are the
same test here.

**The shrinkage rows are the point of this notebook.** Raw estimates carry a
spread across predicted-vol quintiles. The lowest-vol names are understated
(they look safe partly by luck) and the highest-vol names are overstated.
Shrinkage should flatten that spread; on the sample it went 0.096 → 0.001. This
is the bias an optimizer exploits, since it buys the names whose risk looks lowest.

---
## Stage 3 — Estimate Δ(t)

In [ ]:
if REBUILD_SPECIFIC or not db.table_exists(engine, config.TBL_SPECIFIC_RISK):
    D = specific_risk.estimate(U, EX, dates)
    vra = D.attrs["vra"]
    rebuilt = True
else:
    D = db.read_table(config.TBL_SPECIFIC_RISK, engine)
    D["forecast_date"] = pd.to_datetime(D["forecast_date"])
    vra = D.groupby("forecast_date")["vra_lambda"].first().reset_index()
    rebuilt = False
    print(f"📂 loaded {len(D):,} rows from {config.TBL_SPECIFIC_RISK}")

### Verify before believing

In [ ]:
live = EX[EX["eom"].isin(dates) & (pd.to_numeric(EX[config.CAP_COL], errors="coerce") > 0)]
per_date = D.groupby("forecast_date").size()
checks = {
    "a forecast at every one of F's dates": set(D["forecast_date"]) == set(dates),
    "every live asset priced at every date": per_date.equals(live.groupby("eom").size().rename_axis("forecast_date")),
    "no zero, negative or missing specific variance": bool((D["specific_var"] > 0).all()),
    "(id, forecast_date) unique": not D.duplicated(["id", "forecast_date"]).any(),
    "shrinkage lowers the cross-sectional dispersion of vol at every date": bool(
        D.dropna(subset=["specific_var_raw"]).groupby("forecast_date").apply(
            lambda g: np.sqrt(g["specific_var_shrunk"]).std() < np.sqrt(g["specific_var_raw"]).std(),
            include_groups=False).all()),
}
for k, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {k}")
sv = np.sqrt(D["specific_var"] * 12) * 100
print(f"\nspecific vol, annualized: median {sv.median():.1f}%, p5 {sv.quantile(.05):.1f}%, "
      f"p95 {sv.quantile(.95):.1f}%, max {sv.max():.1f}%")
print(f"assets per date: {per_date.min()}–{per_date.max()}")
print(f"rows on the size-bucket prior (thin history): {D['specific_var_raw'].isna().sum()} "
      f"({D['specific_var_raw'].isna().mean():.2%})")

---
## Stage 4 — What shrinkage did

**Left:** raw against shrunk vol on the latest date. Points below the diagonal at
the top and above it at the bottom are the shrinkage at work. **Right:** out of
sample, the bias statistic by predicted-vol quintile for each stage. A flat line
means the model is equally right about safe-looking and risky-looking names.

In [ ]:
last = D[D["forecast_date"] == dates[-1]]
m_final = realized(D)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
rv = np.sqrt(last["specific_var_raw"] * 12) * 100
sh = np.sqrt(last["specific_var_shrunk"] * 12) * 100
ax[0].scatter(rv, sh, s=16, color=INK, alpha=.7, lw=0)
lim = [0, np.nanmax([rv.max(), sh.max()]) * 1.05]
ax[0].plot(lim, lim, color=MUTED, lw=1)
ax[0].set(title=f"Raw vs shrunk specific vol, {dates[-1].date()}", xlabel="raw, %/yr",
          ylabel="shrunk, %/yr", xlim=lim, ylim=lim)

for col, name in STAGE_NAMES.items():
    z = m_final["u_next"] / np.sqrt(m_final[col])
    ok = np.isfinite(z)
    q = pd.qcut(m_final.loc[ok, col], 5, labels=False)
    b = z[ok].groupby(q).std()
    ax[1].plot(range(1, 6), b.values, color=STAGE_COLORS[col], lw=2, marker="o", label=name)
    ax[1].annotate(f" {name}", (5, b.values[-1]), color=STAGE_COLORS[col], fontsize=9, va="center")
ax[1].axhline(1, color=MUTED, lw=1)
ax[1].set(title="Out-of-sample bias by predicted-vol quintile", xlabel="quintile (1 = lowest predicted vol)",
          ylabel="bias statistic", xticks=range(1, 6), xlim=(.7, 5.8))
for a in ax:
    tidy(a)
plt.tight_layout()
plt.show()

---
## Stage 5 — Specific risk level through time, and the regime adjustment

In [ ]:
lvl = D.groupby("forecast_date").apply(
    lambda g: np.sqrt(np.average(g["specific_var"], weights=g["cap"]) * 12) * 100, include_groups=False)
med = D.groupby("forecast_date")["specific_var"].median().pipe(lambda s: np.sqrt(s * 12) * 100)

fig, ax = plt.subplots(2, 1, figsize=(13, 5.6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
ax[0].plot(med.index, med, color=INK, lw=1.8)
ax[0].annotate(" median name", (med.index[-1], med.iloc[-1]), color=INK, fontsize=9, va="center")
ax[0].plot(lvl.index, lvl, color=ACCENT, lw=1.8)
ax[0].annotate(" cap-weighted", (lvl.index[-1], lvl.iloc[-1]), color=ACCENT, fontsize=9, va="center")
ax[0].set(title="Forecast specific vol", ylabel="%/yr")
ax[0].margins(x=.08)
ax[1].plot(vra["forecast_date"], vra["vra_lambda"], color=INK, lw=1.5)
ax[1].axhline(1, color=MUTED, lw=1)
ax[1].set(title="Specific volatility regime adjustment λ_S", ylabel="λ_S")
for a in ax:
    tidy(a)
plt.tight_layout()
plt.show()
print("Small caps carry far more specific risk than large caps. That gap is why the")
print("cap-weighted level sits well below the median name, and it is the reason the")
print("WLS in 02 used √cap weights.")

---
## Stage 6 — Thin histories and the extremes

The sample had no entrants. Here names list continuously, and every name with
fewer than `SPECIFIC_MIN_OBS` residuals in its window takes its size-bucket
prior outright. The share of names on the prior is the number to watch: under
the R1000 profile it should stay small, because entrants have R3000 history
before they join. Then the extremes at the latest date, where a data error would
show up first.

In [ ]:
prior_share = D.groupby("forecast_date")["specific_var_raw"].apply(lambda s: s.isna().mean())
fig, ax = plt.subplots(figsize=(13, 2.8))
ax.plot(prior_share.index, prior_share * 100, color=INK, lw=1.8)
ax.set(title=f"Share of names on the size-bucket prior (< {config.SPECIFIC_MIN_OBS} residuals)", ylabel="%")
tidy(ax)
plt.tight_layout()
plt.show()
print(f"median {prior_share.median():.1%}, max {prior_share.max():.1%}")

conm = EX.drop_duplicates("id").set_index("id")["conm"]
last = D[D["forecast_date"] == dates[-1]].join(conm.rename("conm"), on="id")
last["spec_vol_%"] = np.sqrt(last["specific_var"] * 12) * 100
cols = ["conm", "cap", "size_bucket", "n_obs", "shrink_weight", "spec_vol_%"]
print(f"\nhighest specific vol at {dates[-1].date()}:")
display(last.nlargest(10, "spec_vol_%")[cols].round(3))
print("lowest:")
display(last.nsmallest(10, "spec_vol_%")[cols].round(3))

---
## Stage 7 — Write the artifact

In [ ]:
if rebuilt:
    out = D[["id", "forecast_date", "specific_var", "specific_var_raw", "specific_var_shrunk",
             "shrink_weight", "size_bucket", "n_obs", "cap", "vra_lambda"]]
    db.write_table(out, config.TBL_SPECIFIC_RISK, engine, unique_on=["id", "forecast_date"])
else:
    print("resumed from the table — nothing written")

---
## What 05 does next

Assembles Σ(t) = X(t) F(t) X(t)ᵀ + Δ(t) without ever forming the N×N matrix, then
runs the out-of-sample bias tests at the asset and portfolio level. That includes
the min-variance portfolio of **assets**, which is what 07 actually optimizes.